# SARD - Search and Rescue Human Detection (YOLOv8m)

**Goal**: Train YOLOv8m on the SARD dataset to achieve >90% mAP@0.5 for human detection in search and rescue scenarios.

**Dataset**: 5,755 images (4,041 train / 1,144 val / 570 test), single class `human`, YOLO format, 640x640.

## Kaggle Setup
1. Upload `search-and-rescue/` folder as a Kaggle dataset named **"search-and-rescue"**
2. Enable **GPU T4 or P100** accelerator (Settings → Accelerator)
3. Attach the dataset to this notebook (Add Data → Your Datasets → search-and-rescue)
4. Run all cells

The dataset setup cell auto-detects whether Kaggle mounted the files at `/kaggle/input/search-and-rescue/train/...` or `/kaggle/input/search-and-rescue/search-and-rescue/train/...`.

In [ ]:
!pip install ultralytics --quiet

In [ ]:
import torch
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 1. Configure Dataset

In [ ]:
# Auto-detect dataset directory across Kaggle mount paths and local development
candidate_dirs = [
    "/kaggle/input/search-and-rescue",
    "/kaggle/input/search-and-rescue/search-and-rescue",
    "/kaggle/input/datasets/search-and-rescue",
    "/kaggle/input/datasets",
    os.path.expanduser("~/SARD/search-and-rescue"),
]
DATASET_DIR = next(
    (path for path in candidate_dirs if os.path.isdir(os.path.join(path, "train", "images"))),
    None,
)
if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not find train/images. Check the attached Kaggle dataset path under /kaggle/input."
    )

print(f"Using dataset directory: {DATASET_DIR}")

# Verify dataset structure
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
        raise FileNotFoundError(f"Missing folders for {split}: {img_dir} or {lbl_dir}")
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f"{split}: {n_img} images, {n_lbl} labels")

# Create data.yaml with absolute paths
data_yaml = {
    "train": os.path.join(DATASET_DIR, "train", "images"),
    "val": os.path.join(DATASET_DIR, "valid", "images"),
    "test": os.path.join(DATASET_DIR, "test", "images"),
    "nc": 1,
    "names": ["human"],
}

yaml_path = "/kaggle/working/data.yaml" if os.path.isdir("/kaggle/working") else os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\ndata.yaml saved to {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))

## 2. Load YOLOv8m Model

In [ ]:
model = YOLO("yolov8m.pt")
print("Loaded YOLOv8m pretrained on COCO (includes 'person' class)")

## 3. Train with Augmentation

Aggressive augmentation tuned for this dataset:
- **Small objects** (63.5% have area < 0.01): mosaic, copy_paste, scale
- **Low objects-per-image** (avg 1.51): copy_paste, mixup create multi-object composites
- **Aerial perspectives**: rotation, flipud, shear, perspective
- **Diverse terrain**: HSV color shifts
- **close_mosaic=15**: fine-tunes on real images in last 15 epochs

In [ ]:
results = model.train(
    data=yaml_path,

    # Core settings
    imgsz=640,
    epochs=150,
    batch=16,
    patience=30,
    workers=8,
    device=0,
    pretrained=True,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_bias_lr=0.05,

    # HSV augmentation
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,

    # Geometric augmentation
    degrees=15.0,
    translate=0.2,
    scale=0.7,
    shear=5.0,
    perspective=0.0005,
    flipud=0.2,
    fliplr=0.5,

    # Advanced augmentation
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.2,
    erasing=0.4,
    close_mosaic=15,

    # Loss weights
    box=7.5,
    cls=0.5,
    dfl=1.5,

    # Misc
    seed=42,
    deterministic=False,
    project="SARD_Project",
    name="yolov8m_train",
    exist_ok=True,
    amp=True,
    resume=False,
    val=True,
)

## 4. Validate on Validation Set

In [ ]:
model = YOLO("/kaggle/working/SARD_Project/yolov8m_train/weights/best.pt")
metrics = model.val(data=yaml_path, split="val", imgsz=640, batch=16, device=0)

print("\n=== Validation Results ===")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")

## 5. Evaluate on Test Set

In [ ]:
test_metrics = model.val(data=yaml_path, split="test", imgsz=640, batch=16, device=0)

print("\n=== Test Set Results ===")
print(f"mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print(f"Precision:     {test_metrics.box.mp:.4f}")
print(f"Recall:        {test_metrics.box.mr:.4f}")

if test_metrics.box.map50 >= 0.90:
    print("\n TARGET REACHED: mAP@0.5 >= 90%!")
else:
    print(f"\n mAP@0.5 = {test_metrics.box.map50:.4f} (< 90%). Consider:")
    print("  - Increasing epochs to 200, patience to 40")
    print("  - Trying imgsz=1280 with batch=4")
    print("  - Increasing copy_paste to 0.3 and mixup to 0.25")

## 6. Export Best Weights

In [ ]:
best_src = "/kaggle/working/SARD_Project/yolov8m_train/weights/best.pt"
last_src = "/kaggle/working/SARD_Project/yolov8m_train/weights/last.pt"

shutil.copy(best_src, "/kaggle/working/best_sard_yolov8m.pt")
shutil.copy(last_src, "/kaggle/working/last_sard_yolov8m.pt")

print(f"Best weights: {best_src}")
print(f"Size: {os.path.getsize(best_src) / 1e6:.1f} MB")
print("\nDownload 'best_sard_yolov8m.pt' from the Output tab.")

## 7. Sample Predictions

In [ ]:
import glob
import matplotlib.pyplot as plt

test_images = sorted(glob.glob(os.path.join(DATASET_DIR, "test", "images", "*.jpg")))[:9]

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
for i, img_path in enumerate(test_images):
    results = model.predict(img_path, conf=0.25, imgsz=640, device=0, verbose=False)
    annotated = results[0].plot()
    ax = axes[i // 3][i % 3]
    ax.imshow(annotated[..., ::-1])
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/sample_predictions.png", dpi=150)
plt.show()
print("Sample predictions saved.")

## 8. Training Curves

In [ ]:
from IPython.display import Image as IPImage, display

results_dir = "/kaggle/working/SARD_Project/yolov8m_train"
for name in ["results.png", "confusion_matrix.png", "labels.jpg", "P_curve.png", "R_curve.png"]:
    path = os.path.join(results_dir, name)
    if os.path.exists(path):
        print(f"\n--- {name} ---")
        display(IPImage(filename=path, width=800))